# Lesson 10.3 - Edge AI & TinyML (tiny model demo)

## Objectives

- Train a compact neural network classifier.
- Apply dynamic quantization.
- Compare model size and inference behavior before/after compression.


In [ ]:
from __future__ import annotations

import os
import tempfile
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

torch.manual_seed(42)
np.random.seed(42)


## Generate Toy Dataset

We use a small binary classification problem to mimic low-footprint edge inference use cases.


In [ ]:
X, y = make_classification(
    n_samples=4000,
    n_features=16,
    n_informative=10,
    n_redundant=2,
    class_sep=1.0,
    random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)


## Define and Train Small MLP


In [ ]:
class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(16, 24),
            nn.ReLU(),
            nn.Linear(24, 12),
            nn.ReLU(),
            nn.Linear(12, 1),
        )

    def forward(self, x):
        return self.net(x)


model_fp32 = TinyMLP()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_fp32.parameters(), lr=1e-3)

for epoch in range(80):
    model_fp32.train()
    optimizer.zero_grad()
    logits = model_fp32(X_train_t)
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()

model_fp32.eval()
with torch.no_grad():
    preds = (torch.sigmoid(model_fp32(X_test_t)) > 0.5).int().squeeze().numpy()
acc_fp32 = accuracy_score(y_test, preds)
print(f"FP32 test accuracy: {acc_fp32:.4f}")


## Dynamic Quantization

Dynamic quantization converts linear layers to int8-friendly representations for smaller footprint and potentially faster CPU inference.


In [ ]:
model_int8 = torch.quantization.quantize_dynamic(
    model_fp32,
    {nn.Linear},
    dtype=torch.qint8,
)

model_int8.eval()
with torch.no_grad():
    preds_q = (torch.sigmoid(model_int8(X_test_t)) > 0.5).int().squeeze().numpy()
acc_int8 = accuracy_score(y_test, preds_q)
print(f"INT8 dynamic-quantized test accuracy: {acc_int8:.4f}")


In [ ]:
def model_size_bytes(model: nn.Module) -> int:
    with tempfile.NamedTemporaryFile(delete=False) as f:
        torch.save(model.state_dict(), f.name)
        size = os.path.getsize(f.name)
    os.remove(f.name)
    return size

size_fp32 = model_size_bytes(model_fp32)
size_int8 = model_size_bytes(model_int8)

print(f"FP32 state_dict size: {size_fp32/1024:.2f} KB")
print(f"INT8 state_dict size: {size_int8/1024:.2f} KB")
print(f"Size reduction: {(1 - size_int8/size_fp32)*100:.2f}%")


## Connect to Theory

- Training occurs on workstation/cloud.
- Quantization prepares the model for constrained deployment.
- Size/accuracy trade-off is explicit and measurable.

In real TinyML pipelines, model conversion and operator compatibility with device runtime must also be validated.


## Business Case Studies & Exceptions

### Case A: Vibration Monitoring on MCU

Tiny classifiers run on-device to detect abnormal vibration signatures and trigger maintenance alerts before failure.

### Case B: Wake-word Detection

Local keyword spotting reduces latency and preserves privacy by avoiding continuous audio streaming.

### Exceptions

- If model complexity exceeds MCU budget, use edge SoC or cloud-assisted architecture.
- If quantization degrades critical accuracy too much, keep higher-precision model on stronger edge hardware.


## Interview Questions & Answers

1. **What is TinyML?**  
   ML inference on very constrained embedded hardware, often microcontrollers.
2. **Why quantize models?**  
   To shrink model footprint and improve deployment feasibility on edge devices.
3. **What is dynamic quantization in PyTorch?**  
   Runtime quantization of certain layers (e.g., Linear) to int8 operations.
4. **What are core edge constraints?**  
   Memory, compute, power, latency, and thermal budget.
5. **Why might edge be better than cloud?**  
   Lower latency, offline reliability, and privacy.
6. **What metrics should you track after compression?**  
   Accuracy, model size, latency, and energy.
7. **How can TinyML fail in production?**  
   Sensor drift and unseen conditions can degrade performance.
8. **What is a common deployment pattern?**  
   Sensor -> on-device inference -> local action -> cloud telemetry.
9. **When avoid TinyML?**  
   When task requires heavy models or frequent large-scale updates beyond device capacity.
10. **How do you move from prototype to device?**  
   Validate operator support, convert model, integrate firmware loop, benchmark on hardware.
